# Đánh giá full RAG + Critic Agent trên Colab

Chỉ cần sửa tham số ở Cell **THAM SỐ** bên dưới, rồi **Runtime → Run all**.

**Yêu cầu trước khi chạy:**
1. Runtime → Change runtime type → chọn **GPU** (T4 miễn phí là đủ).
2. Đã upload `data/.qdrant/` và `data/ai_vietnamese_embedding_v2_finetuned_final/` lên Google Drive (đúng thư mục điền ở `DRIVE_DATA_DIR`).
3. Đã đưa Knowledge Graph lên Neo4j Aura (điền `NEO4J_URI`/`NEO4J_PASSWORD` đúng của bạn).


In [ ]:
# ============================================================
# THAM SỐ — CHỈNH Ở ĐÂY, KHÔNG CẦN SỬA GÌ Ở CÁC Ô BÊN DƯỚI
# ============================================================

# --- Neo4j Aura (điền thông tin của bạn — KHÔNG commit lại giá trị thật vào git) ---
NEO4J_URI = "neo4j+s://xxxxxxxx.databases.neo4j.io"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = ""

# --- Google Drive: nơi đã upload sẵn data/.qdrant/ và model fine-tune ---
DRIVE_DATA_DIR = "/content/drive/MyDrive/legal_kg_data"

# --- Git repo chứa code ---
GIT_REPO_URL = "https://github.com/PhuIT2503/LEGAL_IT_CHATBOT.git"
REPO_DIR = "LEGAL_IT_CHATBOT"

# --- LLM sinh câu trả lời (Ollama, chạy ngay trong Colab, dùng GPU) ---
OLLAMA_MODEL = "qwen2.5:7b"

# --- Bộ câu hỏi test ---
# vd: "data/eval_testset.jsonl"              (301 câu, đủ 4 nhóm)
#     "data/eval_testset_stratified10.jsonl" (10 câu, nhanh để test thử)
TESTSET_PATH = "data/eval_testset.jsonl"
LIMIT = None                 # None = chạy hết; hoặc số nguyên (vd 10) để test nhanh
OUTPUT_SUFFIX = "_full301"   # hậu tố tên file kết quả — đổi mỗi lần chạy để không ghi đè
MODES = ["naive", "article_expand", "critic"]
RESUME = True                # True = bỏ qua câu đã có sẵn kết quả (chạy lại an toàn sau khi bị ngắt)

# --- Chấm điểm Legal Completeness Rate (LUÔN chạy, dùng Ollama, không cần key) ---
SKIP_RAGAS = False            # True nếu chỉ muốn Completeness Rate, bỏ qua RAGAS hoàn toàn

# --- RAGAS (faithfulness / answer_relevancy / context_precision / answer_correctness) ---
# Chỉ dùng khi SKIP_RAGAS = False. Tự do chọn 1 trong 3 provider:
RAGAS_PROVIDER = "gemini"    # "openai" | "gemini" | "ollama"  (ollama = free, không cần key)
RAGAS_MODEL = None           # None = mặc định của provider (gpt-4o-mini / gemini-1.5-flash / qwen2.5:7b)
OPENAI_API_KEY = ""          # chỉ cần điền nếu RAGAS_PROVIDER == "openai"
GOOGLE_API_KEY = ""          # chỉ cần điền nếu RAGAS_PROVIDER == "gemini"


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Lấy code từ GitHub

In [ ]:
import os

if not os.path.isdir(REPO_DIR):
    !git clone {GIT_REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} đã tồn tại, pull code mới nhất...")
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}


## 3. Copy dữ liệu nặng (.qdrant + model fine-tune) từ Drive

In [ ]:
import os

os.makedirs("data", exist_ok=True)

qdrant_src = os.path.join(DRIVE_DATA_DIR, ".qdrant")
model_src = os.path.join(DRIVE_DATA_DIR, "ai_vietnamese_embedding_v2_finetuned_final")

if not os.path.isdir("data/.qdrant"):
    print("Copy data/.qdrant từ Drive...")
    !cp -r "{qdrant_src}" data/.qdrant
else:
    print("data/.qdrant đã có sẵn, bỏ qua copy.")

if not os.path.isdir("data/ai_vietnamese_embedding_v2_finetuned_final"):
    print("Copy model fine-tune từ Drive (~2.2GB, có thể mất vài phút)...")
    !cp -r "{model_src}" data/ai_vietnamese_embedding_v2_finetuned_final
else:
    print("Model fine-tune đã có sẵn, bỏ qua copy.")

print("Xong.")


## 4. Cài dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q ragas datasets


## 5. Cài + chạy Ollama (dùng GPU Colab)

In [ ]:
import subprocess, time, requests

!curl -fsSL https://ollama.com/install.sh | sh

ollama_proc = subprocess.Popen(["ollama", "serve"])

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        break
    except Exception:
        time.sleep(2)

!ollama pull {OLLAMA_MODEL}
print("Ollama sẵn sàng.")


## 6. Set biến môi trường (Aura + RAGAS key nếu có)

In [ ]:
import os

os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USER"] = NEO4J_USER
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD
os.environ["OLLAMA_MODEL"] = OLLAMA_MODEL
# OLLAMA_BASE_URL không cần set — mặc định localhost:11434, đúng vì Ollama chạy ngay trong Colab VM này

if RAGAS_PROVIDER == "openai" and OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
if RAGAS_PROVIDER == "gemini" and GOOGLE_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("Đã set biến môi trường.")


## 7. Chạy đánh giá (run_evaluation.py) cho từng mode

In [ ]:
for mode in MODES:
    cmd = f'python scripts/run_evaluation.py --mode {mode} --testset {TESTSET_PATH} --output-suffix {OUTPUT_SUFFIX}'
    if LIMIT:
        cmd += f' --limit {LIMIT}'
    if RESUME:
        cmd += ' --resume'
    print(f"\n{'='*80}\n=== Chạy mode={mode} ===\n{'='*80}")
    get_ipython().system(cmd)


## 8. Chấm điểm (Completeness Rate + RAGAS)

In [ ]:
modes_str = " ".join(MODES)
cmd = f'python scripts/score_evaluation.py --modes {modes_str} --suffix {OUTPUT_SUFFIX}'
if SKIP_RAGAS:
    cmd += ' --skip-ragas'
else:
    cmd += f' --ragas-provider {RAGAS_PROVIDER}'
    if RAGAS_MODEL:
        cmd += f' --ragas-model {RAGAS_MODEL}'
get_ipython().system(cmd)


## 9. Tính MRR / nDCG@5 / Context Precision-Recall / Avg context chars

In [ ]:
import json, math

def load_rows(mode):
    path = f"data/eval_results_{mode}{OUTPUT_SUFFIX}.jsonl"
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def mrr_ndcg(rows):
    sum_rr, sum_ndcg = 0.0, 0.0
    for r in rows:
        gt = {x.lower() for x in r["dieu_ids"]}
        ranked = [x.lower() for x in r["retrieved_dieu_ids_ranked"]]
        rank = next((i + 1 for i, d in enumerate(ranked) if d in gt), 0)
        sum_rr += 1 / rank if rank else 0
        dcg = sum(1 / math.log2(i + 2) for i, d in enumerate(ranked[:5]) if d in gt)
        ideal_hits = min(len(gt), 5)
        idcg = sum(1 / math.log2(i + 2) for i in range(ideal_hits))
        sum_ndcg += dcg / idcg if idcg > 0 else 0
    n = len(rows) or 1
    return sum_rr / n, sum_ndcg / n

def precision_recall(rows):
    precisions, recalls = [], []
    for r in rows:
        gt = {x.lower() for x in r["dieu_ids"]}
        final_set = {x.lower() for x in r["retrieved_dieu_ids_ranked"]} | {x.lower() for x in r["graph_fetched_dieu_ids"]}
        hit = len(final_set & gt)
        precisions.append(hit / len(final_set) if final_set else 0)
        recalls.append(min(hit, len(gt)) / len(gt) if gt else 0)
    n = len(rows) or 1
    return sum(precisions) / n, sum(recalls) / n

def avg_context_chars(rows):
    total = sum(sum(len(c) for c in (r.get("retrieved_contexts") or [])) for r in rows)
    return total / (len(rows) or 1)

naive_rows = load_rows("naive")
mrr, ndcg = mrr_ndcg(naive_rows)
print(f"MRR (retrieval chung cho cả {len(MODES)} mode) = {mrr:.3f}, nDCG@5 = {ndcg:.3f}\n")

retrieval_metrics = {}
for mode in MODES:
    rows = load_rows(mode)
    prec, rec = precision_recall(rows)
    chars = avg_context_chars(rows)
    retrieval_metrics[mode] = {"precision": prec, "recall": rec, "avg_context_chars": chars}
    print(f"{mode}: precision={prec:.3f}, recall={rec:.3f}, avg_context_chars={chars:.0f}")


## 10. Bảng tổng hợp cuối cùng

In [ ]:
import csv

summary_path = f"data/eval_summary{OUTPUT_SUFFIX}.csv"
by_mode = {}
with open(summary_path, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        by_mode.setdefault(row["mode"], {})[f"{row['category']}::{row['metric']}"] = row["value"]

def fmt_pct(v):
    try:
        return f"{float(v) * 100:.1f}%"
    except (TypeError, ValueError):
        return "-"

def fmt_num(v):
    try:
        return f"{float(v):.0f}"
    except (TypeError, ValueError):
        return "-"

# Tên hiển thị RÕ RÀNG cho từng chỉ số RAGAS (key thật trong CSV là "ragas_<ten_metric>")
RAGAS_METRIC_LABELS = {
    "ragas_faithfulness": "Faithfulness (RAGAS)",
    "ragas_answer_relevancy": "Answer Relevancy (RAGAS)",
    "ragas_context_precision": "Context Precision (RAGAS)",
    "ragas_answer_correctness": "Answer Correctness (RAGAS)",
}

print(f"\n{'=' * 100}\nBẢNG TỔNG HỢP{OUTPUT_SUFFIX}\n{'=' * 100}")
print(f"MRR = {mrr:.3f} | nDCG@5 = {ndcg:.3f}  (chung cho cả {len(MODES)} mode, đo bước retrieval)\n")

header = f"{'Chỉ số':<32}" + "".join(f"{m:<20}" for m in MODES)
print(header)
print("-" * len(header))

def print_row(label, get_value_fn):
    print(f"{label:<32}" + "".join(f"{get_value_fn(m):<20}" for m in MODES))

# --- Legal Completeness Rate (chỉ số trung tâm của khóa luận) ---
print_row("Completeness Rate", lambda m: fmt_pct(by_mode.get(m, {}).get("ALL::completeness_rate")))

# --- Chi phí token ---
print_row("Avg total tokens/câu", lambda m: fmt_num(by_mode.get(m, {}).get("ALL::avg_total_tokens")))
print_row("Avg final-answer tokens/câu", lambda m: fmt_num(by_mode.get(m, {}).get("ALL::avg_final_answer_tokens")))
print_row("Avg số lệnh gọi LLM", lambda m: fmt_num(by_mode.get(m, {}).get("ALL::avg_llm_calls")))

# --- Context Precision/Recall tự tính (theo Điều, KHÁC với RAGAS context_precision) ---
print_row("Context Precision (theo Điều)", lambda m: f"{retrieval_metrics[m]['precision']:.3f}")
print_row("Context Recall (theo Điều)", lambda m: f"{retrieval_metrics[m]['recall']:.3f}")
print_row("Avg context chars", lambda m: f"{retrieval_metrics[m]['avg_context_chars']:.0f}")

# --- RAGAS (nếu có chạy) — in TÊN RÕ RÀNG cho từng chỉ số, cùng bảng luôn ---
if not SKIP_RAGAS:
    print()
    print(f"--- RAGAS (provider={RAGAS_PROVIDER}{', model=' + RAGAS_MODEL if RAGAS_MODEL else ''}) ---")
    for ragas_key, label in RAGAS_METRIC_LABELS.items():
        print_row(label, lambda m, k=ragas_key: fmt_pct(by_mode.get(m, {}).get(f"ALL::{k}")))


## 11. Sao lưu kết quả về Google Drive (Colab mất hết khi hết phiên)

In [ ]:
import shutil, glob, os

dest = os.path.join(DRIVE_DATA_DIR, "results")
os.makedirs(dest, exist_ok=True)

patterns = [
    f"data/eval_results_*{OUTPUT_SUFFIX}.jsonl",
    f"data/eval_scores_*{OUTPUT_SUFFIX}.csv",
    f"data/eval_summary{OUTPUT_SUFFIX}.csv",
]
for pattern in patterns:
    for f in glob.glob(pattern):
        shutil.copy(f, dest)
        print(f"Đã copy {f} -> {dest}")

print("\nHoàn tất — kết quả đã lưu vào Google Drive, không mất khi hết phiên Colab.")
